### Preparation for JMeter test
- creation of query files
- generation of jmx files
- jdbc drivers install


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets (text) ---
dbutils.widgets.removeAll()

dbutils.widgets.text("dbx_catalog", "", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "", "Databricks Schema")
dbutils.widgets.text("dbx_volume", "", "Volume name (for results)")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v
    
# --- Validation Check (all required) ---

if params["dbx_catalog"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_catalog' is required but is empty.")

if params["dbx_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_schema' is required but is empty.")

if params["dbx_volume"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_volume' is required but is empty.")

In [ ]:
%sh
rm -rf /tpcds
mkdir -p /tpcds

In [ ]:
%sh
set -e
cd /tmp

# --- Cache strategy ---
# Downloads land in $CACHE_DIR (a /tmp subfolder of the unified Volume)
# so they survive across cluster boots and across notebook re-runs. On a
# clean cache the binary is fetched from the Apache CDN; if the CDN no
# longer hosts this version (Apache rotates dlcdn after each minor) we
# fall back to the always-on archive mirror. The OS package install
# (openjdk-11) is left untouched — it's an apt fetch, not cacheable here.
CACHE_DIR=/Volumes/$W_DBX_CATALOG/$W_DBX_SCHEMA/$W_DBX_VOLUME/tmp
mkdir -p "$CACHE_DIR"

JMETER_FILE=apache-jmeter-5.6.3.tgz
JMETER_URL_PRIMARY="https://dlcdn.apache.org/jmeter/binaries/$JMETER_FILE"
JMETER_URL_FALLBACK="https://archive.apache.org/dist/jmeter/binaries/$JMETER_FILE"

if [ -f "$CACHE_DIR/$JMETER_FILE" ]; then
  echo "cache hit: $CACHE_DIR/$JMETER_FILE"
  cp "$CACHE_DIR/$JMETER_FILE" /tmp/jmeter.tgz
else
  echo "cache miss; trying primary mirror"
  if wget --quiet "$JMETER_URL_PRIMARY" -O /tmp/jmeter.tgz; then
    echo "downloaded from dlcdn"
  else
    echo "primary failed; falling back to archive.apache.org"
    wget --quiet "$JMETER_URL_FALLBACK" -O /tmp/jmeter.tgz
  fi
  cp /tmp/jmeter.tgz "$CACHE_DIR/$JMETER_FILE"
  echo "cached: $CACHE_DIR/$JMETER_FILE"
fi
tar -xzf /tmp/jmeter.tgz

# Install Java 11 — Databricks JDBC 3.x and Snowflake JDBC 4.x both need Java 11+,
# but DBR 15.4 ships Java 8 by default. JMeter must run on Java 11 for the JDBC
# drivers to load.
apt-get update -qq
apt-get install -y -qq openjdk-11-jre-headless

# Resolve the Java 11 home (first match)
JAVA11_HOME=$(ls -d /usr/lib/jvm/java-11-openjdk-* 2>/dev/null | head -1)
if [ -z "$JAVA11_HOME" ]; then
  echo "ERROR: openjdk-11 not found after install" >&2
  exit 1
fi
echo "JAVA11_HOME=$JAVA11_HOME"
$JAVA11_HOME/bin/java -version

# Replace the jmeter shim with a wrapper that pins JAVA_HOME=Java 11
cat > /usr/bin/jmeter <<WRAP
#!/bin/bash
export JAVA_HOME=$JAVA11_HOME
exec /tmp/apache-jmeter-5.6.3/bin/jmeter "\$@"
WRAP
chmod +x /usr/bin/jmeter

echo "jmeter wrapper installed:"
cat /usr/bin/jmeter


In [ ]:
%sh 
apt-get update
apt-get install -y unzip

In [ ]:
%sh
set -e

# Cache JAR/zip downloads in $CACHE_DIR so re-runs do not re-fetch from
# vendor S3/Maven. On cache miss the file is fetched once and stashed.
CACHE_DIR=/Volumes/$W_DBX_CATALOG/$W_DBX_SCHEMA/$W_DBX_VOLUME/tmp
mkdir -p "$CACHE_DIR"

rm -rf /mnt/jdbc-drivers
mkdir -p /mnt/jdbc-drivers
cd /mnt/jdbc-drivers

# --- JDBC driver versions ---
DBX_JDBC_VERSION=3.4.4
SF_JDBC_VERSION=4.4.0

# --- Databricks JDBC Driver ---
DBX_FILE=DatabricksJDBC-$DBX_JDBC_VERSION.zip
DBX_URL="https://databricks-bi-artifacts.s3.us-east-2.amazonaws.com/simbaspark-drivers/jdbc/$DBX_JDBC_VERSION/$DBX_FILE"
if [ -f "$CACHE_DIR/$DBX_FILE" ]; then
  echo "cache hit: $DBX_FILE"
  cp "$CACHE_DIR/$DBX_FILE" driver.zip
else
  echo "cache miss; downloading $DBX_URL"
  wget --quiet "$DBX_URL" -O driver.zip
  cp driver.zip "$CACHE_DIR/$DBX_FILE"
  echo "cached: $CACHE_DIR/$DBX_FILE"
fi
JAR_PATH=$(unzip -l driver.zip | grep -v __MACOSX | grep '\.jar$' | awk '{print $4}')
unzip -jqo driver.zip "$JAR_PATH" -d ./
rm driver.zip
# The jar name inside the zip varies by release (DatabricksJDBC.jar, databricks-jdbc-<version>.jar);
# normalize it so the symlink below always resolves.
JAR_NAME=$(basename "$JAR_PATH")
[ "$JAR_NAME" = "DatabricksJDBC.jar" ] || mv "$JAR_NAME" DatabricksJDBC.jar
test -f /mnt/jdbc-drivers/DatabricksJDBC.jar
ln -sf /mnt/jdbc-drivers/DatabricksJDBC.jar /tpcds/db_driver.jar
echo "Databricks JDBC driver: /tpcds/db_driver.jar -> $(readlink /tpcds/db_driver.jar)"

# --- Snowflake JDBC Driver ---
SF_FILE=snowflake-jdbc-$SF_JDBC_VERSION.jar
SF_URL="https://repo1.maven.org/maven2/net/snowflake/snowflake-jdbc/$SF_JDBC_VERSION/$SF_FILE"
if [ -f "$CACHE_DIR/$SF_FILE" ]; then
  echo "cache hit: $SF_FILE"
  cp "$CACHE_DIR/$SF_FILE" snowflake-jdbc.jar
else
  echo "cache miss; downloading $SF_URL"
  wget --quiet "$SF_URL" -O snowflake-jdbc.jar
  cp snowflake-jdbc.jar "$CACHE_DIR/$SF_FILE"
  echo "cached: $CACHE_DIR/$SF_FILE"
fi
test -f /mnt/jdbc-drivers/snowflake-jdbc.jar
ln -sf /mnt/jdbc-drivers/snowflake-jdbc.jar /tpcds/sf_driver.jar
echo "Snowflake JDBC driver: /tpcds/sf_driver.jar -> $(readlink /tpcds/sf_driver.jar)"


In [ ]:
%sh
mkdir -p /Volumes/$W_DBX_CATALOG/$W_DBX_SCHEMA/$W_DBX_VOLUME/results
mkdir -p /Volumes/$W_DBX_CATALOG/$W_DBX_SCHEMA/$W_DBX_VOLUME/logs
mkdir -p /Volumes/$W_DBX_CATALOG/$W_DBX_SCHEMA/$W_DBX_VOLUME/test_files